#### &#128337; Temps previst de dedicació: 25 minuts

# TEMA 4: Treball amb API

> <br>🚨&nbsp; **Atenció**
> 
> <span style="color: #f69;">En aquest nivell B2 explorem conceptes avançats de JavaScript que excedeixen les capacitats d'un notebook. Descarregueu [Visual Studio Code](https://code.visualstudio.com) a la vostra màquina i creeu projectes locals per a cada exemple proposat, incloent els fitxers HTML, CSS i JavaScript que calguin en cada cas. Instal·leu també les extensions [Live Server](https://marketplace.visualstudio.com/items?itemName=ritwickdey.LiveServer) i [Thunder Client](https://marketplace.visualstudio.com/items?itemName=rangav.vscode-thunder-client). La primera us permetrà executar el vostre codi en un servidor web local des de Visual Studio Code. La segona us resultarà especialment útil per poder treballar més còmodament amb diferents API a mida que estudieu les unitats d'aquest Tema 4.</span>
> <br><br>

## Unitat 4: Autenticació mitjançant JWT

A les unitats anteriors hem explorat com dur a terme operacions amb API mitjançant peticions `GET`, `POST`, `PUT`, `PATCH` i `DELETE`. Tot i que algunes API ofereixen informació pública que no requereix una autorització especial, altres poden contenir dades de naturalesa privada que, per tant, exigeixen mecanismes de seguretat per controlar que només aquells usuaris amb els permisos adequats poden tenir accés a les dades. En aquesta unitat ens centrarem en l'autenticació mitjançant **JSON Web Tokens** (JWT), una metodologia clau per garantir que només els usuaris autoritzats puguin accedir a informació sensible. 

Malgrat que hi ha altres metodologies d'autenticació, JWT és una de les més utilitzades actualment. La seva popularitat es deu al fet que és un mètode senzill d'implementar i que no requereix emmagatzematge d'informació en el servidor. A més, és un mètode que es pot utilitzar en qualsevol tipus d'aplicació, bé sigui web, mòbil o d'escriptori. En aquest sentit, tot i que la "J" de JWT fa referència a JSON i la part "JS" de JSON fa referència a JavaScript, tant JSON com JWT no són eines exclusives de JavaScript, sinó que es poden utilitzar en qualsevol llenguatge de programació.


### Estructura bàsica d'un JWT

Un JWT és un token que es genera en el servidor i que s'envia al client perquè aquest l'utilitzi en les peticions que faci al servidor. El token s'envia a la capçalera de la petició HTTP, de forma que el servidor pugui comprovar que l'usuari està autoritzat per dur a terme l'operació sol·licitada.

És important tenir present que un JWT no és més que un objecte JSON que conté informació sobre l'usuari i que es firma digitalment per garantir que no s'ha alterat. Aquesta firma es genera utilitzant un algoritme de xifratge que utilitza una clau secreta que només coneix el servidor. Per tant, el token el genera sempre el servidor. D'aquesta manera, si l'usuari intenta modificar el token, la firma no coincidirà i el servidor refusarà el token. 

Per exemple, un JWT descodificat pot tenir l'aspecte següent:

In [ ]:
// Header
{
  "alg": "HS256",
  "typ": "JWT"
}

// Payload
{
  "sub": "1234567890",
  "name": "John Doe",
  "iat": 1516239022,
  "admin": true
}

// Signature
HMAC_SHA256(
  secret_key_that_only_server_knows,
  base64urlEncoding(header) + '.' +
  base64urlEncoding(payload)
)

El token està format per diverses parts: _header_ o capçalera, _payload_ o càrrega útil i _signature_ o firma. El payload és la part de les dades transmeses que conté la informació útil, és a dir, la informació essencial que es vol enviar o processar. En concret, el payload d'un JWT només inclou informació de l'usuari (com el seu ID) i una petita col·lecció de _claims_, que no són una altra cosa que parells clau-valor que contenen informació addicional sobre aquest. Per exemple, un claim pot ser el rol de l'usuari (administrador, editor, etc.). Un altre claim pot servir per indicar la data d'expiració del token. Un altre més pot contenir informació sobre el dispositiu des del qual s'ha iniciat sessió. En realitat, és l'equip de desenvolupament qui decideix, amb total llibertat, quants claims contindrà el payload i quina informació emmagatzemarà cada un d'aquests. 

El header, en canvi, conté informació sobre el tipus de token i l'algoritme de xifratge utilitzat.

Finalment, la firma és una cadena de caràcteres que s'utilitza per verificar la integritat del token. Inclou el header i el payload, i se xifra utilitzant l'algoritme de xifratge especificat en el header i la clau secreta del servidor.

En resum, un JWT és un objecte JSON que es firma digitalment i que es pot utilitzar per transmetre informació entre dues parts de forma segura i senzilla, ja que una vegada firmat el token consisteix en un simple rast de caràcteres com el que es pot veure a continuació:


In [ ]:
eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.
eyJzdWIiOiIxMjM0NTY3ODkwIiwibmFtZSI6IkpvaG4gRG9lIiwiaWF0IjoxNTE2MjM5MDIyLCJhZG1pbiI6dHJ1ZX0.
gtsOdcOAOHe9qExUU3w8ECM8C6R-it50uFvQhusVTCY

Podem observar que inclou tres parts separades per punts. La primera part és el header, la segona és el payload i la tercera és la firma. Tot i que pot semblar que el token està codificat i ofereix una certa seguretat, la veritat és que qualsevol el pot descodificar i llegir-ne el contingut. Per tant, mai s'ha d'emmagatzemar informació sensible en el payload.

> <br>🚨&nbsp; **Atenció**
> 
> La clau secreta s'ha de desar al servidor i mai s'ha de compartir amb el client. Si el client tingués accés a la clau secreta, podria generar els seus propis tokens i accedir a la informació privada d'altres usuaris.
>
> Insistim en aquest punt clau: el payload mai ha de contenir informació sensible. Tot i que el token està firmat digitalment, qualsevol en pot llegir el contingut, per exemple en pàgines web com [jwt.io](https://jwt.io). Per tant, mai inclogueu informació com contrasenyes o números de targeta de crèdit en el payload.
> <br><br>


### Flux d'autenticació i autorització mitjançant JWT

L'autenticació i autorització mitjançant JWT funcionen seguint aquest esquema:

1. **Autenticació de l'usuari** – En primer lloc, l'usuari s'ha d'autenticar a l'aplicació, generalment a través d'un formulari d'inici de sessió. Quan s'envien aquestes dades, l'aplicació verifica les credencials introduïdes contra la base de dades o un servei d'autenticació. És a dir, aquest primer pas no és una altra cosa que un login tradicional.

2. **Generació de JWT** – Un cop s'ha autenticat l'usuari, el servidor genera un JWT amb l'estructura que hem comentat anteriorment. Com sigui que la firma del token es genera utilitzant un algoritme de xifratge que utilitza una clau secreta que només coneix el servidor, si l'usuari intenta modificar el token, la firma no coincidirà i el servidor el refusarà. El JWT és un objecte JSON que es firma digitalment i que es pot utilitzar per transmetre informació entre dues parts de forma segura.

3. **Enviament del JWT al client** – El servidor envia el JWT al client, que l'emmagatzema, per exemple en l'emmagatzematge local del navegador o en una galeta.

4. **Ús del JWT en sol·licituds posteriors** – Quan el client fa sol·licituds posteriors al servidor, ja no necessita passar per un nou sistema d'autenticació, sinó que es limita a incloure el JWT a les capçaleres de la sol·licitud, generalment a la capçalera `Authorization`.

5. **Validació del JWT en el servidor** – El servidor verifica la validesa del JWT a cada sol·licitud. Per a això, comprova la firma digital per assegurar-se que el token no s'ha alterat i que és autèntic. Recordem que el JWT s'ha generat partint d'una clau secreta que només coneix el servidor. Per tant, el servidor pot verificar l'autenticitat del token comprovant que la firma coincideix amb la que es genera utilitzant la clau secreta. Si no coincideix, el token es refusa.

6. **Autorització** – Una vegada validat el JWT, el servidor utilitza la informació continguda en el token per determinar si l'usuari té permís per dur a terme l'acció sol·licitada. Aquí és on entren en joc tots els claims que s'han inclòs en el payload del token. Per exemple, si l'usuari té el rol d'administrador, el servidor li permetrà fer operacions que no estaran disponibles per a la resta d'usuaris. Si l'usuari no té permís per dur a terme l'acció sol·licitada, el servidor retornarà un error. El client final (l'usuari) pot arribar a llegir el contingut del token i saber quins claims té assignats, però no pot modificar aquesta informació, perquè no disposa de la clau secreta del servidor. Així, el servidor pot confiar en la informació que conté el token i utilitzar-la per determinar si l'usuari té permís per dur a terme l'acció sol·licitada.

Aquest flux permet que les aplicacions web manegin l'autenticació i autorització d'una manera segura i eficient, sense necessitat de verificar les credencials de l'usuari amb cada sol·licitud, cosa que millora de manera notable l'experiència de l'usuari.




### Ús de JWT en una aplicació web

Com acabem de veure en el flux anterior, l'ús de JWT des de la perspectiva del client és molt senzill i consisteix en dos passos fonamentals. En primer lloc, el client ha de rebre i emmagatzemar el token. Quan un usuari s'autentica en una aplicació web, el servidor respondrà amb un JWT ja creat i firmat. El client s'ha d'assegurar d'emmagatzemar aquest token per poder-lo utilitzar en properes sol·licituds. En JavaScript el podem emmagatzemar de la manera següent:

In [ ]:
// Pseudo code. Do NOT try to run this code.
fetch('authentication_login_url', { method: 'POST', body: loginData })
  .then(response => response.json())
  .then(data => {
    localStorage.setItem('jwt', data.jwt);
  });

Bàsicament enviem les dades d'usuari al servidor (probablement un nom d'usuari i una contrasenya) i, si l'autenticació és correcta, rebem el token de tipus JWT a la resposta. En aquest exemple, assumim que el token es rep dins d'un objecte que inclou una propietat `jwt`, tot i que això és una decisió de l'equip de desenvolupament del backend, i decidim desar-lo a l'emmagatzematge local del navegador, també amb la clau `jwt`, tot i que en podríem utilitzar qualsevol altra, en tant que això ja és una decisió de l'equip de desenvolupament del frontend.

Posteriorment, hem d'incloure aquest token a les capçaleres de qualsevol sol·licitud que enviem al servidor. Ho farem amb una sintaxi semblant a la següent:

In [ ]:
// Pseudo code. Do NOT try to run this code.
const jwt = localStorage.getItem('jwt');
fetch('api_url', {
  headers: { 'Authorization': 'Bearer ' + jwt }
})
  .then(response => response.json())
  .then(data => console.log(data));

D'aquesta manera, enviem el token a la capçalera `Authorization` de la sol·licitud. El servidor pot llegir aquest token, verificar-ne la validesa i actuar en conseqüència, atorgant l'accés al recurs sol·licitat o denegant-lo, si detecta que l'usuari no té permisos suficients o que el token s'ha modificat des de la seva creació prèvia en el mateix servidor.

### Claims habituals

Tot i que el payload d'un JWT pot contenir qualsevol informació que l'equip de desenvolupament consideri oportuna, hi ha alguns claims que són habituals i que s'utilitzen en la majoria d'aplicacions. A continuació, es mostren alguns dels més comuns:

- `iss` – L'emissor del token. Sol ser el nom de l'aplicació o el domini d'aquesta.
- `sub` – El subjecte del token. Sol ser l'ID de l'usuari.
- `aud` – El destinatari del token. Sol ser el domini de l'aplicació.
- `exp` – La data d'expiració del token. Si no s'especifica, el token no caduca.
- `nbf` – La data a partir de la qual el token és vàlid.
- `iat` – La data en la qual es va emetre el token.
- `jti` – L'ID del token. S'utilitza per evitar que el token es reutilitzi.
  
Tot i que fora dels estàndards, també és habitual incloure altres claims com els següents:

- `role` – El rol de l'usuari. Per exemple, administrador, editor, etc.
- `device` – El dispositiu des del qual s'ha iniciat sessió. Per exemple, mòbil, tauleta, etc.
- `ip` – L'adreça IP des de la qual s'ha iniciat sessió.
- `ua` – L'agent d'usuari (navegador) des del qual s'ha iniciat sessió.

Realment no hi ha una llista de claims tancada, sinó que cada equip de desenvolupament té llibertat per incloure els que consideri oportuns. El més important és que el servidor puegui llegir el token i utilitzar la informació que conté per determinar si l'usuari té permís per dur a terme l'acció sol·licitada i amb quines condicions o circumstàncies.
  